In [1]:
import sys
sys.path.append('..')

from src.parser import parse_fields, extract_block_id, to_timestamp, build_miner, mine_templates

In [2]:
df = parse_fields('../data/raw/HDFS_sample.log')
df['BlockId'] = df['content'].apply(extract_block_id)
df['ts'] = to_timestamp(df)

df.head()

Parsed: 200000 | Skipped: 0


,date,time,pid,level,component,content,BlockId,ts
0,081109,203518,143,INFO,dfs.DataNode$DataXceiver,Receiving block blk_-1608999687919862906 src: ...,blk_-1608999687919862906,2008-11-09 20:35:18
1,081109,203518,35,INFO,dfs.FSNamesystem,BLOCK* NameSystem.allocateBlock: /mnt/hadoop/m...,blk_-1608999687919862906,2008-11-09 20:35:18
2,081109,203519,143,INFO,dfs.DataNode$DataXceiver,Receiving block blk_-1608999687919862906 src: ...,blk_-1608999687919862906,2008-11-09 20:35:19
3,081109,203519,145,INFO,dfs.DataNode$DataXceiver,Receiving block blk_-1608999687919862906 src: ...,blk_-1608999687919862906,2008-11-09 20:35:19
4,081109,203519,145,INFO,dfs.DataNode$PacketResponder,PacketResponder 1 for block blk_-1608999687919...,blk_-1608999687919862906,2008-11-09 20:35:19


In [3]:
print("Rows:", len(df))
print("Levels:", df['level'].value_counts().to_dict())
print("Unique components:", df['component'].nunique())
print("Unique blocks:", df['BlockId'].nunique())
print("Missing block IDs:", df['BlockId'].isna().sum())
print("Time span:", df['ts'].min(), "→", df['ts'].max())

Rows: 200000
Levels: {'INFO': 199992, 'WARN': 8}
Unique components: 8
Unique blocks: 15639
Missing block IDs: 0
Time span: 2008-11-09 20:35:18 → 2008-11-09 21:01:28


In [4]:
miner = build_miner('../drain3.ini')
df = mine_templates(df, miner)

print("Unique templates:", df['EventId'].nunique())

Unique templates: 25


In [5]:
df['EventTemplate'].value_counts()

EventTemplate
Receiving block <BLK> src: /<IP> dest: /<IP>                                                                                                                                                                            46523
BLOCK* NameSystem.addStoredBlock: blockMap updated: <IP> is added to <BLK> size <NUM>                                                                                                                                   44925
Received block <BLK> of size <NUM> from /<IP>                                                                                                                                                                           44877
PacketResponder <NUM> for block <BLK> terminating                                                                                                                                                                       27683
PacketResponder <NUM> for block <BLK> <*>                                                         

In [6]:
df.to_csv('../data/parsed/hdfs_parsed.csv', index=False)
miner.save_state('after initial parsing')
print("Saved")

Saved
